#**Atividade para Casa**


---

**Objetivo:** Implementar um classificador de spam e entender as métricas de avaliação.


---

###**Instruções:**
1. Baixar o notebook exemplo disponível no repositório
2. Executar o código de regressão logística (dataset de e-mails)
3. **Testar mudanças:**
  - Alterar a proporção de treino/teste
  - Mudar o limite de decisão (se quiser testar)
4. **Responder no notebook:**
  - Qual foi a acurácia, a precisão e recall obtidos?
  - O que cada métrica significa nesse contexto (spam)?
  - Para um filtro de spam, qual métrica você acha mais importante? Por quê?
5. Subir o notebook respondido na pasta da semana 10 do repositório.

#**Bibliotecas**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

#**Carregamento do Dataset**

In [2]:
# 1. Carregar o dataset
# Usamos 'latin-1' pois esse dataset específico costuma ter caracteres especiais
df = pd.read_csv('spam.csv', encoding='latin-1')

# Manter apenas as colunas úteis e renomeá-las
df = df[['v1', 'v2']]
df.columns = ['label', 'mensagem']

# Converter labels para valores binários (0 = ham/normal, 1 = spam)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# 2. Vetorização do texto (Transformar texto em números)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['mensagem'])
y = df['label']

#**Mudança 1: Alterar a Proporção de Treino/Teste**

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# 3. Treinar o modelo de Regressão Logística
model = LogisticRegression()
model.fit(X_train, y_train)

# 4. Fazer previsões (Limite de decisão padrão: 0.5)
y_pred_padrao = model.predict(X_test)

print("--- RESULTADOS COM LIMITE DE DECISÃO PADRÃO (0.5) ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_padrao):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_padrao):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_padrao):.4f}\n")

--- RESULTADOS COM LIMITE DE DECISÃO PADRÃO (0.5) ---
Acurácia: 0.9575
Precisão: 1.0000
Recall:   0.6758



#**Mudança 2: Mudar o Limite de Decisão (Threshold)**

In [4]:
probabilidades = model.predict_proba(X_test)[:, 1] # Pega a probabilidade da classe 1 (spam)
novo_limite = 0.70
y_pred_novo_limite = (probabilidades >= novo_limite).astype(int)

print(f"--- RESULTADOS COM NOVO LIMITE DE DECISÃO ({novo_limite}) ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_novo_limite):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_novo_limite):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_novo_limite):.4f}")

--- RESULTADOS COM NOVO LIMITE DE DECISÃO (0.7) ---
Acurácia: 0.9205
Precisão: 1.0000
Recall:   0.3927


## **Respostas e Análise das Métricas**

### **1. Qual foi a acurácia, a precisão e recall obtidos?**

**Com o Limite de Decisão Padrão (0.5):**
* **Acurácia:** 0.9575 (95.75%)
* **Precisão:** 1.0000 (100.00%)
* **Recall:** 0.6758 (67.58%)

**Com o Novo Limite de Decisão (0.7):**
* **Acurácia:** 0.9205 (92.05%)
* **Precisão:** 1.0000 (100.00%)
* **Recall:** 0.3927 (39.27%)

---

### **2. O que cada métrica significa nesse contexto (spam)?**

* **Acurácia:** É a taxa de acerto geral do filtro. Ela indica que, de todas as mensagens que testamos, o modelo classificou corretamente 95.75% delas (sejam elas spams bloqueados ou e-mails normais entregues).
* **Precisão:** Mede a confiabilidade do alerta de spam. Uma precisão de 1.0000 (100%) significa que **todas** as mensagens que o modelo rotulou como "SPAM" eram espams de verdade. Nenhum e-mail legítimo (ham) foi classificado incorretamente como spam (zero Falsos Positivos).
* **Recall (Revocação):** Mede a capacidade do modelo de encontrar os spams. O recall de 67.58% indica que o modelo conseguiu capturar cerca de 67% dos spams reais, mas deixou passar aproximadamente 33% deles diretamente para a caixa de entrada do usuário (Falsos Negativos).

---

### **3. Para um filtro de spam, qual métrica você acha mais importante? Por quê?**

A **Precisão** é a métrica mais importante para um filtro de spam, pois no cenário de filtragem de e-mails, o custo de um *Falso Positivo* é maior do que o custo de um *Falso Negativo*. Se o **Recall** for baixo, o usuário sofre apenas o incômodo visual de ter que apagar manualmente alguns spams. No entanto, se a **Precisão** for baixa, o usuário pode perder e-mails cruciais, gerando prejuízos reais.

**Análise do Experimento:**
Nos nossos testes, o modelo padrão (0.5) já atingiu a precisão perfeita de 100%. Quando aumentamos o limite para 0.7, a precisão continuou em 100% (pois já era máxima), mas o recall desabou de 67.58% para 39.27%, fazendo com que o filtro deixasse passar mais de 60% dos spams existentes. Portanto, para este dataset, o limite padrão de **0.5** se mostrou muito mais equilibrado e eficiente.